# Simmer — a real AI photo for every recipe
Generate images with Stable Diffusion in a **Colab GPU runtime**, then save them to Cloudinary and your recipe database. This notebook makes no OpenAI API calls. GPU availability depends on Colab.

1. Deploy the included site update first.
2. In Colab choose **Runtime → Change runtime type → GPU**. Run cells 1–4 to review one preview.
3. Run cells 5–6 to generate and publish the whole collection. Credentials are requested privately and are never saved in this notebook.

Already-generated images are skipped. If the GPU session stops, reconnect and run again: the images already published stay saved. Existing photos remain visible until their replacement is successfully generated and uploaded. AI illustrations can differ from the actual cooked meal.


In [ ]:
#@title 1. Prepare the cloud GPU
import importlib.metadata
from pathlib import Path
import subprocess
import sys

try:
    import google.colab
except ImportError:
    raise RuntimeError('Open this notebook in Google Colab. It is designed to run there.') from None
if sys.version_info < (3, 11):
    raise RuntimeError('Select a current Python 3 runtime in Colab, then run this cell again.')

# Preserve Colab's preinstalled, driver-compatible PyTorch and torchvision.
runtime_pins = []
for package in ('torch', 'torchvision'):
    try:
        runtime_pins.append(package + '==' + importlib.metadata.version(package))
    except importlib.metadata.PackageNotFoundError:
        raise RuntimeError('Select a standard GPU runtime that includes PyTorch and torchvision.') from None
constraints = Path('/content/simmer-runtime-constraints.txt')
constraints.write_text('\n'.join(runtime_pins) + '\n')
packages = [
    'fastai==2.8.12', 'diffusers==0.42.1', 'transformers==5.22.2',
    'accelerate==1.15.0', 'safetensors==0.8.0',
    'cloudinary==1.47.0', 'pymongo==4.19.0',
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', '-c', str(constraints), *packages])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('No GPU is assigned. Choose Runtime → Change runtime type → GPU. If unavailable, try later.')
print('Ready. Colab GPU:', torch.cuda.get_device_name(0))
print('This is the cloud runtime, not your own computer.')


In [ ]:
#@title 2. Load the live recipe collection
SITE_URL = 'https://simmer-recipe-app.onrender.com' #@param {type:"string"}
import copy, requests
from urllib.parse import urlparse
if urlparse(SITE_URL).scheme != 'https':
    raise ValueError('Use the full HTTPS address of your deployed site.')

def load_catalog():
    try:
        response = requests.get(SITE_URL.rstrip('/') + '/api/media/catalog', timeout=90)
        response.raise_for_status()
        values = response.json()['recipes']
        if not isinstance(values, list) or not values:
            raise ValueError('Empty catalog')
        if not all(isinstance(r.get('generation', {}).get('prompt'), str) and r.get('id') for r in values):
            raise ValueError('Missing generation details')
        return values
    except Exception:
        raise RuntimeError('Could not load the generation catalog. Deploy this site update first, check SITE_URL, and retry.') from None

def needs_photo(recipe):
    return not (recipe.get('imageGenerated') and recipe.get('image') and (
        not recipe.get('imageRecipeHash') or recipe['imageRecipeHash'] == recipe['generation']['recipeHash']))

recipes = load_catalog()
recipes_by_id = {r['id']: r for r in recipes}
print('Recipes:', len(recipes), '• Need an AI photo:', sum(needs_photo(r) for r in recipes))
print('First recipe IDs:', ', '.join(r['id'] for r in recipes[:12]))


In [ ]:
#@title 3. Load the Stable Diffusion engine
"""Pretrained Stable Diffusion, executed through a real fastai Learner inference loop.
No model training or fine-tuning is claimed by this application.
"""
import io
import os
import threading
from dataclasses import dataclass
import numpy as np
import torch
from torch import nn
from PIL import Image
from fastai.data.core import DataLoaders, TfmdDL
from fastai.learner import Learner

@dataclass(frozen=True)
class Generation:
    prompt: str
    seed: int = 42
    steps: int = 25

class DiffusionBatch(nn.Module):
    """Adapt Diffusers' image output to fastai's tensor prediction/gathering protocol."""
    def __init__(self, pipeline, request: Generation, device: str):
        super().__init__()
        self.anchor = nn.Parameter(torch.zeros(1, device=device), requires_grad=False)
        self.pipeline = pipeline
        self.request = request
        self.device_name = device
        self.blocked = False

    def forward(self, _batch):
        generator_device = 'cuda' if self.device_name == 'cuda' else 'cpu'
        result = self.pipeline(
            prompt=self.request.prompt,
            negative_prompt='text, watermark, logo, person, hands, illustration, deformed food, blurry, raw uncooked poultry',
            num_inference_steps=self.request.steps,
            guidance_scale=7.0,
            width=512, height=512,
            generator=torch.Generator(device=generator_device).manual_seed(self.request.seed),
            output_type='np',
        )
        self.blocked = any(result.nsfw_content_detected or [])
        # Prediction tensors are gathered by Learner.get_preds rather than returning a fabricated result.
        return torch.from_numpy(np.asarray(result.images, dtype=np.float32)).permute(0, 3, 1, 2).to(self.anchor.device)

def fastai_generate(pipeline, request: Generation, device='cpu') -> bytes:
    model = DiffusionBatch(pipeline, request, device)
    # Input-only batches: this is inference, not an optimization/training dataset.
    dl = TfmdDL([(torch.zeros(1),)], bs=1, num_workers=0, device=torch.device(device))
    dls = DataLoaders(dl, dl, device=torch.device(device))
    learn = Learner(dls, model, loss_func=lambda *_: torch.tensor(0.0), default_cbs=False)
    predictions, _ = learn.get_preds(dl=dl, act=lambda x: x, reorder=False)
    if model.blocked:
        raise ValueError('The model safety checker declined this image.')
    array = predictions[0].permute(1, 2, 0).clamp(0, 1).mul(255).round().to(torch.uint8).cpu().numpy()
    output = io.BytesIO()
    Image.fromarray(array).save(output, format='PNG')
    return output.getvalue()

class ImageEngine:
    def __init__(self):
        self.pipeline = None
        self.device = 'cpu'
        self.loading_error = None
        self._load_lock = threading.Lock()

    def load(self):
        with self._load_lock:
            if self.pipeline is not None:
                return
            from diffusers import StableDiffusionPipeline
            requested = os.getenv('DEVICE', 'auto')
            self.device = ('cuda' if torch.cuda.is_available() else 'cpu') if requested == 'auto' else requested
            if self.device == 'cuda' and not torch.cuda.is_available():
                raise RuntimeError('DEVICE=cuda was requested but CUDA is unavailable.')
            if self.device not in ('cpu', 'cuda', 'mps'):
                raise RuntimeError('DEVICE must be auto, cpu, cuda or mps.')
            dtype = torch.float16 if self.device == 'cuda' else torch.float32
            self.pipeline = StableDiffusionPipeline.from_pretrained(
                os.getenv('MODEL_ID', 'stable-diffusion-v1-5/stable-diffusion-v1-5'),
                torch_dtype=dtype, use_safetensors=True, token=os.getenv('HF_TOKEN') or None,
            ).to(self.device)
            self.pipeline.enable_attention_slicing()
            self.pipeline.set_progress_bar_config(disable=True)

    def generate(self, request: Generation) -> bytes:
        self.load()
        return fastai_generate(self.pipeline, request, self.device)


In [ ]:
#@title 4. Generate and review a recipe preview
RECIPE_ID = 'avocado-toast' #@param {type:"string"}
STEPS = 30 #@param {type:"slider", min:20, max:40, step:1}
import os, time
from IPython.display import Image as NotebookImage, display
if RECIPE_ID not in recipes_by_id:
    raise ValueError('Choose an exact recipe ID from the catalog.')
if not torch.cuda.is_available():
    raise RuntimeError('Choose a Colab GPU runtime and run the preparation cells again.')
os.environ['DEVICE'] = 'cuda'
os.environ['MODEL_ID'] = 'stable-diffusion-v1-5/stable-diffusion-v1-5'
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
    del hf_token
except Exception:
    pass
if 'simmer_engine' not in globals():
    simmer_engine = ImageEngine()
selected_recipe = copy.deepcopy(recipes_by_id[RECIPE_ID])
preview_png = None
preview_key = None
print('Creating:', selected_recipe['name'])
try:
    preview_png = simmer_engine.generate(Generation(selected_recipe['generation']['prompt'][:1600], selected_recipe['generation']['seed'], STEPS))
except Exception as error:
    raise RuntimeError('Preview generation failed (' + type(error).__name__ + '). Check GPU memory or model access and retry.') from None
preview_key = (selected_recipe['id'], selected_recipe['generation']['recipeHash'])
display(NotebookImage(data=preview_png))
print('Preview only. No photo has been replaced yet.')


In [ ]:
#@title 5. Load the resumable batch publisher
import hashlib, io, uuid

RECIPE_FIELDS = ('name', 'description', 'ingredients', 'steps')

def publish_generated_photo(recipe, png_bytes, collection, upload, destroy, folder='simmer'):
    if not isinstance(png_bytes, bytes) or not png_bytes.startswith(b'\x89PNG\r\n\x1a\n') or len(png_bytes) > 10 * 1024 * 1024:
        raise ValueError('The model did not return a valid PNG under 10 MB.')
    current = collection.find_one({'id': recipe['id']})
    if current is None:
        raise ValueError('This recipe does not exist in the selected database.')
    if current.get('image') != recipe.get('image'):
        raise ValueError('The photo changed since the catalog was loaded. Refresh the catalog before retrying.')
    for field in RECIPE_FIELDS:
        if current.get(field) != recipe.get(field):
            raise ValueError('The recipe changed. Refresh the catalog to generate its updated dish.')
    result = upload(io.BytesIO(png_bytes), folder=folder.strip('/') + '/generated',
                    public_id=recipe['id'] + '-' + uuid.uuid4().hex,
                    resource_type='image', format='png', overwrite=False)
    url, public_id = result.get('secure_url', ''), result.get('public_id', '')
    if urlparse(url).scheme != 'https' or not urlparse(url).netloc or not public_id:
        raise RuntimeError('Cloud storage did not return a valid image address.')
    changes = {'image': url, 'imagePublicId': public_id, 'imageProvider': 'cloudinary',
               'imageGenerated': True, 'imageRecipeHash': recipe['generation']['recipeHash'],
               'imageSeed': recipe['generation']['seed'], 'imageModel': os.environ.get('MODEL_ID', ''),
               'imageCredit': 'AI-generated illustration', 'imageSource': '', 'imageLicense': '', 'imageLicenseUrl': ''}
    guard = {'id': recipe['id'], 'image': recipe.get('image')}
    guard.update({field: recipe.get(field) for field in RECIPE_FIELDS})
    try:
        updated = collection.update_one(guard, {'$set': changes}, upsert=False)
    except Exception:
        # Keep the upload if the database may already have committed it.
        raise RuntimeError('Could not confirm the database update. Refresh the catalog before retrying.') from None
    if updated.matched_count != 1:
        try:
            destroy(public_id, resource_type='image', invalidate=True)
        except Exception:
            pass
        raise ValueError('The recipe or its photo changed during publishing. The newer version was kept.')
    return changes

def run_photo_batch(catalog, engine, collection, upload, destroy, folder='simmer', steps=30,
                    preview=None, preview_for=None, limit=0):
    candidates = [r for r in catalog if needs_photo(r)]
    if limit:
        candidates = candidates[:limit]
    summary = {'saved': [], 'failed': [], 'skipped': len(catalog) - sum(needs_photo(r) for r in catalog)}
    for index, recipe in enumerate(candidates, 1):
        print(str(index) + '/' + str(len(candidates)) + ' — ' + recipe['name'], flush=True)
        try:
            key = (recipe['id'], recipe['generation']['recipeHash'])
            image = preview if preview is not None and preview_for == key else engine.generate(
                Generation(recipe['generation']['prompt'][:1600], recipe['generation']['seed'], steps))
            changes = publish_generated_photo(recipe, image, collection, upload, destroy, folder)
            recipe.update(changes)
            summary['saved'].append(recipe['id'])
            print('  Saved.', flush=True)
        except Exception as error:
            # Do not print raw provider/database exceptions: they may contain credentials.
            summary['failed'].append({'recipeId': recipe['id'], 'type': type(error).__name__})
            print('  Not saved (' + type(error).__name__ + '). Earlier photos are kept; rerun to retry.', flush=True)
            if type(error).__name__ in ('OutOfMemoryError', 'ServerSelectionTimeoutError', 'AuthenticationError', 'AuthorizationError'):
                print('  Stopping so you can fix the connection or GPU before continuing.', flush=True)
                break
    return summary


In [ ]:
#@title 6. Generate and publish the collection
# Run after inspecting the preview above. This replaces existing stock photos.
MAX_RECIPES = 0 #@param {type:"integer"}
# 0 = every recipe that still needs an AI photo; use 3 for a small first batch.
CLOUDINARY_FOLDER = 'simmer' #@param {type:"string"}
import getpass, ipaddress
import cloudinary, cloudinary.uploader
from pymongo import MongoClient
if not isinstance(MAX_RECIPES, int) or isinstance(MAX_RECIPES, bool) or MAX_RECIPES < 0:
    raise ValueError('MAX_RECIPES must be zero or a positive whole number.')
if not globals().get('preview_png'):
    raise RuntimeError('Generate and review the preview in cell 4 first.')

def private_value(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
    except Exception:
        value = None
    if not value:
        value = getpass.getpass(name + ' (private): ')
    if not isinstance(value, str) or not value.strip():
        raise ValueError('Missing ' + name + '.')
    return value.strip()

# Reload so a resumed notebook sees photos already saved by earlier runs.
recipes = load_catalog()
remaining = sum(needs_photo(r) for r in recipes)
print(remaining, 'recipes still need an AI photo.')
if not remaining:
    print('All recipes already have generated photos. Nothing to do.')
else:
    try:
        ip = requests.get('https://api.ipify.org', timeout=12)
        ip.raise_for_status()
        address = ipaddress.ip_address(ip.text.strip())
        print('Current Colab address for Atlas Network Access:', str(address) + ('/32' if address.version == 4 else '/128'))
    except Exception:
        print('Could not detect the runtime address automatically.')
    mongo_uri = private_value('MONGODB_URI')
    if not mongo_uri.startswith(('mongodb://', 'mongodb+srv://')):
        raise ValueError('Use your complete working MongoDB connection string.')
    cloud_name = private_value('CLOUDINARY_CLOUD_NAME')
    api_key = private_value('CLOUDINARY_API_KEY')
    api_secret = private_value('CLOUDINARY_API_SECRET')
    client = None
    try:
        client = MongoClient(mongo_uri, serverSelectionTimeoutMS=20000, connectTimeoutMS=15000, socketTimeoutMS=20000)
        client.admin.command('ping')
        database = client.get_default_database(default='simmer')
        cloudinary.config(cloud_name=cloud_name, api_key=api_key, api_secret=api_secret, secure=True)
        batch_summary = run_photo_batch(recipes, simmer_engine, database.recipes,
            cloudinary.uploader.upload, cloudinary.uploader.destroy,
            CLOUDINARY_FOLDER, STEPS, preview_png, preview_key, MAX_RECIPES)
        print('Finished — saved:', len(batch_summary['saved']), '• already generated:', batch_summary['skipped'],
              '• not saved:', len(batch_summary['failed']))
        print('Refresh your site to see the generated images. Rerun this cell to retry recipes without AI photos.')
    except Exception as error:
        raise RuntimeError('Cloud publishing could not finish (' + type(error).__name__ + '). Check your private credentials and Atlas network access, then retry. Photos already saved remain available.') from None
    finally:
        if client is not None:
            client.close()
        del mongo_uri, api_key, api_secret, cloud_name
